# PADA-3DACB — preprocesamiento reproducible en Kaggle

Este notebook **clona e instala el repositorio** [`AlejoPatigno/PADA-3DACB`](https://github.com/AlejoPatigno/PADA-3DACB) y usa directamente sus módulos para:

1. descubrir y etiquetar OASIS/ADNI;
2. ejecutar el preprocesamiento MRI canónico a tensores `(1, 128, 128, 128)`;
3. preparar CerebrA como atlas discreto de 102 ROI;
4. precomputar los conceptos regionales canónicos en las dos direcciones `ADNI→OASIS` y `OASIS→ADNI`, ajustando cada normalizador **solo con CN del dominio fuente**.

El registro N4 + rígido + afín de ADNI al referente OASIS se orquesta aquí con SimpleITK. SimpleITK usa CPU; activar GPU en Kaggle no acelera de forma material este paso. No se descargan datasets: añada como entradas de Kaggle `cerebra`, `oasis-1-shinohara` y `adnidataset`.

> Para una prueba rápida deje `SMOKE=True`. Para procesar todo, cambie a `False`.

In [15]:
# 1) Parámetros del experimento
from pathlib import Path

REPOSITORY_URL = "https://github.com/AlejoPatigno/PADA-3DACB.git"
REPOSITORY_REF = "main"
REPO = Path("/kaggle/working/PADA-3DACB")
INPUT = Path("/kaggle/input/datasets")
OUTPUT = Path("/kaggle/working/pada3dacb_preprocessed")

CEREBRA_ROOT = INPUT / "alejopatio/cerebra"
OASIS_ROOT = INPUT / "ninadaithal/oasis-1-shinohara"
ADNI_ROOT = INPUT / "sanjukaggling/adnidataset"

SMOKE = False
SMOKE_SUBJECTS_PER_COHORT = 3
OVERWRITE = False
SEED = 42
TARGET_SHAPE = (128, 128, 128)

BINARY_CLASSES = ("CN", "Impaired")
CLASS_TO_INDEX = {
    "CN": 0,
    "Impaired": 1,
}

for root in (CEREBRA_ROOT, OASIS_ROOT, ADNI_ROOT):
    if not root.exists():
        raise FileNotFoundError(f"Falta el dataset de Kaggle: {root}")
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Entradas verificadas; salida:", OUTPUT)

Entradas verificadas; salida: /kaggle/working/pada3dacb_preprocessed


In [16]:
# 2) Importar el repositorio: clonar, fijar revisión e instalar
import shutil, subprocess, sys

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[full]"], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("PADA-3DACB importado en", commit)

From https://github.com/AlejoPatigno/PADA-3DACB
 * branch            main       -> FETCH_HEAD
HEAD is now at 431fc6f feat(publication): freeze experiments and add ablation framework


PADA-3DACB importado en 431fc6fad88b82d1c38a2354035794b4140dbde2


In [17]:
import sys
import subprocess
from pathlib import Path
import importlib
# 3) Importaciones canónicas desde PADA-3DACB
import hashlib, json, os, re, time
from dataclasses import asdict

import nibabel as nib
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn.functional as F

REPO = Path("/kaggle/working/PADA-3DACB").resolve()
SRC = REPO / "src"

if not (SRC / "pada3dacb").is_dir():
    raise FileNotFoundError(f"No se encontró el paquete en: {SRC / 'pada3dacb'}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)],
    check=True,
)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

importlib.invalidate_caches()

from pada3dacb.artifacts.atlas import (
    AtlasConfig,
    AtlasROIManager,
    export_atlas_metadata,
)
from pada3dacb.artifacts.concepts import (
    ConceptTargetConfig,
    extract_tissue_loss_proxy,
    fit_concept_normalizer,
)
from pada3dacb.data.inventories import (
    discover_and_select,
    extract_adni_subject_id,
)
from pada3dacb.data.preprocessing import (
    DiscoveryConfig,
    ExecutionConfig,
    PreprocessingConfig,
    PreprocessingPaths,
    PreprocessingRunConfig,
    process_scan,
    save_preprocessing_reports,
)

import pada3dacb

print("Módulos cargados desde:", Path(pada3dacb.__file__).resolve())

Módulos cargados desde: /kaggle/working/PADA-3DACB/src/pada3dacb/__init__.py


## Descubrimiento de archivos

Se eligen de forma determinista el CSV de OASIS que contiene `ID` y `CDR`, el volumen de etiquetas CerebrA y los scans OASIS mediante el inventario del propio repositorio.

In [18]:
import re
from pathlib import Path

import pandas as pd

from pada3dacb.data.inventories import *

OASIS_VISIT_RE = re.compile(r"(OAS1_\d{4}_MR\d+)", re.IGNORECASE)


def discover_oasis1_scans(
    root: Path,
    metadata_csv: Path,
    supported_extensions,
):
    metadata = pd.read_csv(metadata_csv)

    normalized_columns = {
        str(column).strip().lower(): column
        for column in metadata.columns
    }

    id_column = next(
        (
            normalized_columns[name]
            for name in ("id", "subject id", "subject_id", "subject")
            if name in normalized_columns
        ),
        None,
    )
    cdr_column = normalized_columns.get("cdr")

    if id_column is None or cdr_column is None:
        raise ValueError(
            f"El CSV debe contener ID y CDR. Columnas encontradas: "
            f"{metadata.columns.tolist()}"
        )

    label_by_visit = {}

    for _, row in metadata.iterrows():
        match = OASIS_VISIT_RE.search(str(row[id_column]))
        if match is None:
            continue

        visit_id = match.group(1).upper()
        cdr = pd.to_numeric(row[cdr_column], errors="coerce")

        if pd.isna(cdr) or float(cdr) < 0:
            continue

        label_by_visit[visit_id] = (
            "CN"
            if float(cdr) == 0.0
            else "Impaired"
        )

    supported_extensions = tuple(
        extension.lower() for extension in supported_extensions
    )

    candidates_by_visit = {}

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        path_lower = str(path).lower()

        if not path_lower.endswith(supported_extensions):
            continue

        # Evita duplicar Analyze 7.5 seleccionando .img y descartando .hdr.
        if path_lower.endswith(".hdr"):
            continue

        match = OASIS_VISIT_RE.search(str(path))
        if match is None:
            continue

        visit_id = match.group(1).upper()

        if visit_id not in label_by_visit:
            continue

        filename = path.name.lower()

        if any(
            token in filename
            for token in ("seg", "aseg", "label", "atlas", "roi")
        ):
            continue

        candidates_by_visit.setdefault(visit_id, []).append(path)

    def scan_priority(path: Path):
        name = path.name.lower()

        score = 0
        if "mprage" in name or "mpr-" in name or "_mpr" in name:
            score -= 10
        if "t1" in name:
            score -= 8
        if "t88" in name:
            score -= 6
        if "masked" in name:
            score -= 4
        if "brain" in name:
            score -= 2
        if path.name.lower().endswith(".nii.gz"):
            score -= 2
        elif path.suffix.lower() == ".nii":
            score -= 1

        return score, str(path)

    selected_scans = []

    for visit_id, paths in sorted(candidates_by_visit.items()):
        ordered_paths = sorted(set(paths), key=scan_priority)
        selected_path = ordered_paths[0]

        selected_scans.append(
            SelectedScan(
                subject_id=visit_id,
                cohort="OASIS",
                class_label=label_by_visit[visit_id],
                selected_path=selected_path,
                all_paths=ordered_paths,
                selection_rule="oasis1_visit_id_then_mprage_t1_priority",
                excluded_paths=ordered_paths[1:],
            )
        )

    return selected_scans


def find_oasis_metadata(root: Path) -> Path:
    for path in sorted(root.rglob("*.csv")):
        try:
            cols = {str(c).strip().lower() for c in pd.read_csv(path, nrows=2).columns}
            if "cdr" in cols and cols.intersection({"id", "subject id", "subject_id", "subject"}):
                return path
        except Exception:
            pass
    raise FileNotFoundError("No se encontró un CSV OASIS con columnas ID y CDR")

def find_cerebra_labels(root: Path) -> Path:
    candidates = []
    for pattern in ("*.mnc", "*.nii.gz", "*.nii"):
        candidates += list(root.rglob(pattern))
    candidates = [p for p in candidates if any(t in p.name.lower() for t in ("cerebra", "label", "atlas"))]
    if not candidates:
        raise FileNotFoundError("No se encontró el volumen de etiquetas CerebrA")
    return sorted(candidates, key=lambda p: ("label" not in p.name.lower(), len(str(p)), str(p)))[0]

OASIS_CSV = find_oasis_metadata(OASIS_ROOT)
oasis_scans = discover_oasis1_scans(
    root=Path(OASIS_ROOT),
    metadata_csv=Path(OASIS_CSV),
    supported_extensions=DiscoveryConfig().supported_extensions,
)

if not oasis_scans:
    raise RuntimeError(
        "No se descubrieron MRI OASIS asociadas a visitas válidas del CSV."
    )

if SMOKE:
    n_cn = max(1, SMOKE_SUBJECTS_PER_COHORT // 2)
    n_impaired = max(
        1,
        SMOKE_SUBJECTS_PER_COHORT - n_cn,
    )

    oasis_cn = [
        scan
        for scan in oasis_scans
        if scan.class_label == "CN"
    ][:n_cn]

    oasis_impaired = [
        scan
        for scan in oasis_scans
        if scan.class_label == "Impaired"
    ][:n_impaired]

    oasis_scans = oasis_cn + oasis_impaired

reference_scan = next(
    (
        scan for scan in oasis_scans
        if scan.class_label == "CN"
    ),
    oasis_scans[0],
)

print(f"Scans OASIS descubiertos: {len(oasis_scans)}")
print(f"Referencia OASIS: {reference_scan.selected_path}")
print(
    pd.Series(
        [scan.class_label for scan in oasis_scans]
    ).value_counts()
)



print(OASIS_CSV)
CEREBRA_LABELS = find_cerebra_labels(CEREBRA_ROOT)


print("CSV OASIS:", OASIS_CSV)
print("CerebrA:", CEREBRA_LABELS)
print("Referente OASIS:", reference_scan.selected_path)
print("OASIS seleccionados:", len(oasis_scans))

Scans OASIS descubiertos: 436
Referencia OASIS: /kaggle/input/datasets/ninadaithal/oasis-1-shinohara/oasis/OASIS/OAS1_0001_MR1_mpr_n4_anon_sbj_111_normalised.nii
CN    336
AD    100
Name: count, dtype: int64
/kaggle/input/datasets/ninadaithal/oasis-1-shinohara/oasis_cross-sectional.csv
CSV OASIS: /kaggle/input/datasets/ninadaithal/oasis-1-shinohara/oasis_cross-sectional.csv
CerebrA: /kaggle/input/datasets/alejopatio/cerebra/mni_icbm152_CerebrA_tal_nlin_sym_09c.mnc
Referente OASIS: /kaggle/input/datasets/ninadaithal/oasis-1-shinohara/oasis/OASIS/OAS1_0001_MR1_mpr_n4_anon_sbj_111_normalised.nii
OASIS seleccionados: 436


## Registro espacial de ADNI

Cada serie DICOM se identifica por `SeriesInstanceUID`. Se excluyen localizadores y series no T1; se prioriza MPRAGE. ADNI se registra al referente OASIS mediante N4, registro rígido y afín con información mutua de Mattes. El atlas y las MRI comparten así el mismo espacio antes del reescalado canónico del repositorio.

In [19]:
ADNI_ID = re.compile(r"\d{3}_S_\d{4}")

def dicom_tag(path: str, tag: str) -> str:
    reader = sitk.ImageFileReader()
    reader.SetFileName(path); reader.LoadPrivateTagsOn(); reader.ReadImageInformation()
    return reader.GetMetaData(tag) if reader.HasMetaDataKey(tag) else ""

def enumerate_adni_series(root: Path) -> pd.DataFrame:
    rows = []
    dirs = sorted({p.parent for p in root.rglob("*.dcm")})
    for directory in dirs:
        for uid in sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(directory)) or []:
            files = list(sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(directory), uid))
            if not files: continue
            first = files[0]
            text = " ".join([dicom_tag(first, "0008|103e"), dicom_tag(first, "0018|1030"), dicom_tag(first, "0008|0008")]).lower()
            modality = dicom_tag(first, "0008|0060").upper()
            match = ADNI_ID.search(str(directory))
            subject = match.group(0) if match else extract_adni_subject_id(directory)
            label = next((p.name.upper() for p in [directory, *directory.parents] if p.name.upper() in {"CN","MCI","AD"}), None)
            is_localizer = any(t in text for t in ("localizer", "scout", "survey"))
            is_t1 = modality in ("", "MR") and any(t in text for t in ("mprage", "mp-rage", "t1", "spgr")) and not is_localizer
            rows.append(dict(subject_id=subject, class_label=label, directory=str(directory), series_uid=uid,
                             slice_count=len(files), description=text, is_t1=is_t1,
                             priority=0 if any(t in text for t in ("mprage", "mp-rage")) else 1))
    frame = pd.DataFrame(rows)
    if frame.empty: raise RuntimeError("No se encontraron series DICOM en ADNI")
    frame = frame[frame.is_t1 & frame.class_label.notna()].copy()
    if frame.empty: raise RuntimeError("No quedó ninguna serie ADNI T1/MPRAGE etiquetada")
    return (frame.sort_values(["subject_id","priority","slice_count","directory"], ascending=[True,True,False,True])
                 .drop_duplicates("subject_id", keep="first").reset_index(drop=True))

def read_series(row) -> sitk.Image:
    files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(row.directory, row.series_uid)
    reader = sitk.ImageSeriesReader(); reader.SetFileNames(files)
    return sitk.Cast(reader.Execute(), sitk.sitkFloat32)

def n4(image: sitk.Image) -> sitk.Image:
    mask = sitk.OtsuThreshold(image, 0, 1, 200)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([50, 50, 30, 20])
    corrector.SetConvergenceThreshold(1e-7)
    return corrector.Execute(image, mask)

def registration_method(seed: int = SEED) -> sitk.ImageRegistrationMethod:
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.20, seed)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsGradientDescent(learningRate=1.0, numberOfIterations=200,
                                      convergenceMinimumValue=1e-6, convergenceWindowSize=10)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4,2,1]); reg.SetSmoothingSigmasPerLevel([2,1,0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    return reg

def register_affine(moving: sitk.Image, fixed: sitk.Image) -> tuple[sitk.Image, sitk.Transform, dict]:
    moving = n4(moving); fixed_f = n4(fixed)
    rigid0 = sitk.CenteredTransformInitializer(fixed_f, moving, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.GEOMETRY)
    rigid_reg = registration_method(); rigid_reg.SetInitialTransform(rigid0, inPlace=False)
    rigid = rigid_reg.Execute(fixed_f, moving)
    affine0 = sitk.AffineTransform(3); affine0.SetCenter(rigid.GetFixedParameters()[:3])
    affine_reg = registration_method(); affine_reg.SetInitialTransform(affine0, inPlace=False)
    affine_reg.SetMovingInitialTransform(rigid)
    affine = affine_reg.Execute(fixed_f, moving)
    composite = sitk.CompositeTransform(3); composite.AddTransform(rigid); composite.AddTransform(affine)
    out = sitk.Resample(moving, fixed, composite, sitk.sitkLinear, 0.0, sitk.sitkFloat32)
    stop = f"{rigid_reg.GetOptimizerStopConditionDescription()} | {affine_reg.GetOptimizerStopConditionDescription()}"
    if "maximum number of iterations" in stop.lower():
        raise RuntimeError("Registro sin convergencia: " + stop)
    qc = {"rigid_metric": float(rigid_reg.GetMetricValue()), "affine_metric": float(affine_reg.GetMetricValue()), "stop": stop}
    return out, composite, qc

In [ ]:
# Ejecutar el registro ADNI (reiniciable)
fixed = sitk.ReadImage(str(reference_scan.selected_path), sitk.sitkFloat32)
adni_inventory = enumerate_adni_series(ADNI_ROOT)
if SMOKE:
    n_cn = max(1, SMOKE_SUBJECTS_PER_COHORT // 2)
    n_impaired = max(
        1,
        SMOKE_SUBJECTS_PER_COHORT - n_cn,
    )

    adni_cn = adni_inventory[
        adni_inventory["binary_class_label"].eq("CN")
    ].head(n_cn)

    adni_impaired = adni_inventory[
        adni_inventory["binary_class_label"].eq("Impaired")
    ].head(n_impaired)

    adni_inventory = (
        pd.concat(
            [adni_cn, adni_impaired],
            ignore_index=True,
        )
        .drop_duplicates("subject_id")
        .reset_index(drop=True)
    )
    
adni_inventory["original_class_label"] = (
    adni_inventory["class_label"]
)

adni_inventory["binary_class_label"] = (
    adni_inventory["class_label"].map(
        {
            "CN": "CN",
            "MCI": "Impaired",
            "AD": "Impaired",
        }
    )
)

adni_inventory = (
    adni_inventory[
        adni_inventory["binary_class_label"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

registered_root = OUTPUT / "registered_adni"
registration_rows = []
for row in adni_inventory.itertuples(index=False):
    repository_folder = (
        "CN"
        if row.binary_class_label == "CN"
        else "AD"
    )
    
    out = (
        registered_root
        / repository_folder
        / f"{row.subject_id}.nii.gz"
    )
    tfm = out.with_suffix("").with_suffix(".tfm")
    out.parent.mkdir(parents=True, exist_ok=True)
    try:
        if OVERWRITE or not out.exists():
            registered, transform, qc = register_affine(read_series(row), fixed)
            sitk.WriteImage(registered, str(out)); sitk.WriteTransform(transform, str(tfm))
        else:
            qc = {"status": "reused"}
        registration_rows.append(
            {
                **row._asdict(),
                "original_class_label": row.class_label,
                "harmonized_class_label": row.binary_class_label,
                "registered_path": str(out),
                "status": "OK",
                "qc": json.dumps(qc),
            }
        )
    except Exception as exc:
        registration_rows.append({**row._asdict(), "registered_path": None, "status": "FAILED", "qc": str(exc)})
    print(row.subject_id, registration_rows[-1]["status"])

registration_df = pd.DataFrame(registration_rows)
registration_df.to_csv(OUTPUT / "adni_registration_manifest.csv", index=False)
if not registration_df.status.eq("OK").any():
    raise RuntimeError("Ninguna serie ADNI superó el registro")

ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000396364



002_S_0413 FAILED
002_S_0559 FAILED


ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182



002_S_0619 FAILED


ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182



...

ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00039697



002_S_1261 FAILED
002_S_1280 FAILED
005_S_0221 FAILED
005_S_0223 FAILED


ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.0002



005_S_0324 FAILED
005_S_0448 FAILED


ImageSeriesReader (0x3e9976a0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000299394



## Preparar CerebrA y ejecutar el preprocesamiento del repositorio

In [ ]:
def validate_binary_classes(labels, cohort):
    counts = (
        pd.Series(labels)
        .value_counts()
        .reindex(BINARY_CLASSES, fill_value=0)
    )

    if (counts == 0).any():
        raise RuntimeError(
            f"{cohort} no contiene ambas clases binarias:\n"
            f"{counts}"
        )

    unknown = set(labels) - set(BINARY_CLASSES)
    if unknown:
        raise RuntimeError(
            f"{cohort} contiene etiquetas no armonizadas: "
            f"{sorted(unknown)}"
        )

    print(f"\n{cohort}\n{counts}")


validate_binary_classes(
    [scan.class_label for scan in oasis_scans],
    "OASIS",
)

validate_binary_classes(
    [scan.class_label for scan in adni_scans],
    "ADNI",
)

In [ ]:
# CerebrA: remuestreo físico NN al referente y reescalado NN a 128³
atlas_raw = sitk.ReadImage(str(CEREBRA_LABELS))
atlas_ref = sitk.Resample(atlas_raw, fixed, sitk.Transform(3, sitk.sitkIdentity), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt16)
# Se guarda primero con la geometría del referente y se aplica la misma
# canonicalización RAS que usa el preprocesamiento NIfTI del repositorio.
atlas_reference_grid = OUTPUT / "cerebra_reference_grid.nii.gz"
sitk.WriteImage(atlas_ref, str(atlas_reference_grid))
atlas_xyz = np.rint(nib.as_closest_canonical(nib.load(str(atlas_reference_grid))).get_fdata()).astype(np.int32)
labels_before = set(np.unique(atlas_xyz).tolist())
if not set(range(1,103)).issubset(labels_before):
    raise ValueError(f"CerebrA no conserva las ROI 1..102; faltan {sorted(set(range(1,103))-labels_before)}")
atlas_t = torch.from_numpy(atlas_xyz.astype(np.float32))[None,None]
atlas_128 = F.interpolate(atlas_t, size=TARGET_SHAPE, mode="nearest")[0,0].numpy().astype(np.int16)
if set(np.unique(atlas_128)) != set(range(103)):
    raise ValueError("El remuestreo hizo desaparecer etiquetas CerebrA")
prepared_atlas = OUTPUT / "cerebra_128.nii.gz"
nib.save(nib.Nifti1Image(atlas_128, np.eye(4)), prepared_atlas)
atlas_manager = AtlasROIManager(prepared_atlas, AtlasConfig(label_values=list(range(1,103)), expected_num_rois=102))
export_atlas_metadata(atlas_manager, OUTPUT / "atlas")
print(atlas_manager.summary())

In [ ]:
# Configuraciones y ejecución canónica de PADA-3DACB
def make_cfg(cohort: str, input_root: Path, output_root: Path, metadata_csv: Path | None = None):
    return PreprocessingRunConfig(
        preprocessing=PreprocessingConfig(target_shape=TARGET_SHAPE, overwrite=OVERWRITE, resume=not OVERWRITE,
                                          continue_on_error=True, compute_source_hash=True),
        discovery=DiscoveryConfig(),
        data=PreprocessingPaths(cohort=cohort, input_root=input_root, metadata_csv=metadata_csv, output_root=output_root),
        execution=ExecutionConfig(seed=SEED, number_of_workers=1),
    )

def same_grid(path: Path, reference: sitk.Image, tol=1e-4) -> bool:
    image = sitk.ReadImage(str(path))
    return (image.GetSize() == reference.GetSize() and
            np.allclose(image.GetSpacing(), reference.GetSpacing(), atol=tol, rtol=0) and
            np.allclose(image.GetOrigin(), reference.GetOrigin(), atol=tol, rtol=0) and
            np.allclose(image.GetDirection(), reference.GetDirection(), atol=tol, rtol=0))

oasis_accepted = [s for s in oasis_scans if same_grid(s.selected_path, fixed)]
if not oasis_accepted:
    raise RuntimeError("Ningún OASIS comparte la cuadrícula física del referente")

oasis_cfg = make_cfg("OASIS", OASIS_ROOT, OUTPUT / "model_ready" / "OASIS", OASIS_CSV)
oasis_records = [process_scan(scan, oasis_cfg) for scan in oasis_accepted]
save_preprocessing_reports(oasis_cfg.data.output_root, oasis_cfg, oasis_records)

adni_cfg = make_cfg("ADNI", registered_root, OUTPUT / "model_ready" / "ADNI")
adni_scans = discover_and_select("ADNI", registered_root, adni_cfg.discovery.supported_extensions)
for scan in adni_scans:
    scan.class_label = (
        "CN"
        if scan.class_label == "CN"
        else "Impaired"
    )
    
if SMOKE: adni_scans = adni_scans[:SMOKE_SUBJECTS_PER_COHORT]
adni_records = [process_scan(scan, adni_cfg) for scan in adni_scans]
save_preprocessing_reports(adni_cfg.data.output_root, adni_cfg, adni_records)

records = oasis_records + adni_records
manifest = pd.DataFrame([asdict(r) for r in records])
manifest = manifest[manifest.status.isin(["PROCESSED", "SKIPPED_VALID"])].copy()
manifest["derivative_path"] = manifest.output_path
manifest.to_csv(OUTPUT / "model_ready_manifest.csv", index=False)
print(manifest.groupby(["cohort", "class_label"]).size())

## Precomputar conceptos mediante PADA-3DACB

La función `extract_tissue_loss_proxy` y `fit_concept_normalizer` proceden del repositorio. Para evitar fuga de etiquetas, en cada dirección el normalizador se ajusta exclusivamente con los sujetos CN del dominio fuente; las etiquetas del destino no intervienen en el cálculo.

In [ ]:
def load_tensor(path):
    obj = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(obj, dict):
        obj = next(obj[k] for k in ("x","image","mri","tensor","volume") if k in obj)
    if tuple(obj.shape) != (1, *TARGET_SHAPE):
        raise ValueError(f"Forma incompatible {tuple(obj.shape)} en {path}")
    return obj.float().contiguous()

def directional_concepts(frame: pd.DataFrame, source: str, target: str) -> pd.DataFrame:
    source_df = frame[frame.cohort.eq(source)].copy()
    target_df = frame[frame.cohort.eq(target)].copy()
    if source_df.empty or target_df.empty:
        raise ValueError(f"Faltan sujetos para {source}→{target}")
    cfg = ConceptTargetConfig(normal_class_name="CN")
    source_proxy = np.stack([extract_tissue_loss_proxy(load_tensor(p), atlas_manager, cfg) for p in source_df.derivative_path])
    target_proxy = np.stack([extract_tissue_loss_proxy(load_tensor(p), atlas_manager, cfg) for p in target_df.derivative_path])
    normalizer = fit_concept_normalizer(source_proxy, source_df.class_label.tolist(), normal_label="CN",
                                        roi_labels=atlas_manager.label_values, cohorts=[source]*len(source_df))
    direction_root = OUTPUT / "concepts" / f"{source}_to_{target}"
    direction_root.mkdir(parents=True, exist_ok=True)
    normalizer.save(direction_root / "source_cn_normalizer.json")
    rows = []
    # La etiqueta target no se lee ni se guarda en los artefactos de adaptación.
    for domain, domain_df, values in (("source", source_df, normalizer.transform(source_proxy)),
                                      ("target", target_df.drop(columns=["class_label"], errors="ignore"), normalizer.transform(target_proxy))):
        for (_, row), vector in zip(domain_df.iterrows(), values, strict=True):
            sid = str(row.subject_hash)
            path = direction_root / domain / f"{sid}_concept.pt"
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(torch.from_numpy(vector).float(), path)
            rows.append({"subject_hash": sid, "cohort": row.cohort, "role": domain,
                         "derivative_path": row.derivative_path, "concept_path": str(path)})
    out = pd.DataFrame(rows)
    out.to_csv(direction_root / "concept_manifest.csv", index=False)
    return out

concept_adni_oasis = directional_concepts(manifest, "ADNI", "OASIS")
concept_oasis_adni = directional_concepts(manifest, "OASIS", "ADNI")
print("ADNI→OASIS:", len(concept_adni_oasis), "vectores")
print("OASIS→ADNI:", len(concept_oasis_adni), "vectores")

In [ ]:
# 7) Validación final y localización de los artefactos
for table in (concept_adni_oasis, concept_oasis_adni):
    for path in table.concept_path:
        vector = torch.load(path, map_location="cpu", weights_only=True)
        assert vector.dtype == torch.float32 and tuple(vector.shape) == (102,) and torch.isfinite(vector).all()

summary = {
    "repository": REPOSITORY_URL,
    "commit": commit,
    "mode": "smoke" if SMOKE else "full",
    "model_ready_subjects": int(len(manifest)),
    "model_ready_by_cohort": manifest.cohort.value_counts().to_dict(),
    "atlas_rois": atlas_manager.K,
    "concept_directions": ["ADNI_to_OASIS", "OASIS_to_ADNI"],
    "registration_backend": "SimpleITK CPU (N4 + rigid + affine)",
}
(OUTPUT / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("\nArtefactos listos en:", OUTPUT)

### Resultado esperado

Los tensores MRI quedan en `model_ready/`, el atlas preparado en `cerebra_128.nii.gz`, los registros y exclusiones en sus manifiestos CSV y los conceptos direccionales en `concepts/`. Para la ejecución completa, cambie únicamente `SMOKE = False` y ejecute todas las celdas.